 # Calculate operator punctuality stats based on Chuuchuu data
 
 This notebook aims to used the merge enriched parquet, to calculate specific outputs about operator's punctuality.

 The overall aim is to compare what we can calculate with Chuuchuu, with operator official statistics (when available)

In [1]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.max_columns", None)   

In [2]:
PARQUET_PATH = "intermediate_outputs/data_chuuchuu_2025_01_to_2025_12_enriched.parquet"

OPERATORS_OF_INTEREST = ["DB", "Eurostar", "Flixtrain", "NS", "OEBB", "Ouigo", "SBB",
                         "SNCB", "SNCF", "Trenitalia", "Trenitalia France", "Westbahn"]

# columns with few distinct values: loaded as pandas categories (much lighter than one string per row)
CATEGORY_COLUMNS = [
    "date", "agency", "routeType", "journey_type", "country","HSR_service", "LD", "night_train", "joint_service",
    "operator_for_punctuality", "normalized_operator", "depart_terminus", "arrivalCancelled_resolved",
]
OTHER_COLUMNS = ["journey_id", "journey_id_country", "is_ambiguous_trip", "sort_time", "arrivalDelay"]

# subset at read time: only the columns of interest, only the stops of the operators of interest,
# without the stops whose journey_type couldn't be resolved ("unknown"),
# so the full 46-column parquet is never loaded in memory
data_chuuchuu = pq.read_table(
    PARQUET_PATH,
    columns=CATEGORY_COLUMNS + OTHER_COLUMNS,
    filters=[
        ("operator_for_punctuality", "in", OPERATORS_OF_INTEREST),
    ],
    read_dictionary=CATEGORY_COLUMNS,
).to_pandas()

# drop the categories no row uses anymore (e.g. the 15 other operators, "unknown" journey_type)
data_chuuchuu[CATEGORY_COLUMNS] = data_chuuchuu[CATEGORY_COLUMNS].apply(lambda col: col.cat.remove_unused_categories())

print(f"{len(data_chuuchuu)} stops, {data_chuuchuu.memory_usage(deep=True).sum() / 1e9:.1f} GB in memory")
print(data_chuuchuu["operator_for_punctuality"].value_counts())
print(data_chuuchuu["journey_type"].value_counts())

78855328 stops, 7.6 GB in memory
operator_for_punctuality
Trenitalia           33597214
NS                   18524980
SNCB                 12552130
SNCF                  6210578
DB                    3419530
SBB                   2808083
OEBB                  1168983
Westbahn               232030
Eurostar               153913
Ouigo                  103378
Flixtrain               58271
Trenitalia France       26238
Name: count, dtype: int64
journey_type
domestic         74533303
international     4300930
unknown             21095
Name: count, dtype: int64


# Adding columns for punctuality

In [3]:
# delayed by strictly more than X min at this stop (arrivalDelay is in seconds): a stop exactly
# X:00 late is not delayed at that threshold. A cancelled stop, or one with no measured delay
# (NaN), is never counted as delayed.
DELAY_THRESHOLDS_MIN = [3, 5, 6, 10, 15, 30, 60]

not_cancelled = data_chuuchuu["arrivalCancelled_resolved"] != "t"
for threshold in DELAY_THRESHOLDS_MIN:
    data_chuuchuu[f"delayed_{threshold}min"] = not_cancelled & (data_chuuchuu["arrivalDelay"] > threshold * 60)

print(data_chuuchuu[[f"delayed_{t}min" for t in DELAY_THRESHOLDS_MIN]].sum())

delayed_3min     12988229
delayed_5min      7877711
delayed_6min      6470234
delayed_10min     3472289
delayed_15min     2093178
delayed_30min      789799
delayed_60min      211043
dtype: int64


# DB

DB is using the following parameter to calculate punctuality :

* Services: Long distance
* Threshold (min): 6
* Measurement: at all stops
* Train or passenger punctuality: train punctuality (<i>pax pctlty also available</i>)
* Account for cancelled trains: No
* Account joint services: cover all collaborations on the German network

NB: for DB, we calculate at the stop level, to be able to compare with their own stat: we can keep rows where we are not sure the stop is terminus or depart. For othter operators, do not forget to do the filter

In [4]:
display(data_chuuchuu[(data_chuuchuu["operator_for_punctuality"]=="DB")&(data_chuuchuu["arrivalCancelled_resolved"]=="f")])


,date,agency,routeType,journey_type,country,HSR_service,LD,night_train,joint_service,operator_for_punctuality,normalized_operator,depart_terminus,arrivalCancelled_resolved,journey_id,journey_id_country,is_ambiguous_trip,sort_time,arrivalDelay,delayed_3min,delayed_5min,delayed_6min,delayed_10min,delayed_15min,delayed_30min,delayed_60min
1361,2025-01-01,DB,ICE,domestic,Germany,Yes,Yes,No,DB,DB,DB,depart,f,DB_ICE_1000_2025-01-01,DB_ICE_1000_2025-01-01_Germany,False,2025-01-01 16:51:00+00:00,NaN,False,False,False,False,False,False,False
1362,2025-01-01,DB,ICE,domestic,Germany,Yes,Yes,No,DB,DB,DB,intermediate,f,DB_ICE_1000_2025-01-01,DB_ICE_1000_2025-01-01_Germany,False,2025-01-01 17:56:00+00:00,0.0,False,False,False,False,False,False,False
1363,2025-01-01,DB,ICE,domestic,Germany,Yes,Yes,No,DB,DB,DB,intermediate,f,DB_ICE_1000_2025-01-01,DB_ICE_1000_2025-01-01_Germany,False,2025-01-01 19:09:00+00:00,60.0,False,False,False,False,False,False,False
1364,2025-01-01,DB,ICE,domestic,Germany,Yes,Yes,No,DB,DB,DB,intermediate,f,DB_ICE_1000_2025-01-01,DB_ICE_1000_2025-01-01_Germany,False,2025-01-01 19:38:00+00:00,120.0,False,False,False,False,False,False,False
1365,2025-01-01,DB,ICE,domestic,Germany,Yes,Yes,No,DB,DB,DB,intermediate,f,DB_ICE_1000_2025-01-01,DB_ICE_1000_2025-01-01_Germany,False,2025-01-01 20:46:00+00:00,240.0,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78855322,2025-12-31,DB,ECE,international,Germany,No,Yes,No,DB,DB,DB,intermediate,f,dedup_ECE_9_2025-12-31,dedup_ECE_9_2025-12-31_Germany,False,2025-12-31 19:14:00+00:00,120.0,False,False,False,False,False,False,False
78855323,2025-12-31,DB,ECE,international,Germany,No,Yes,No,DB,DB,DB,intermediate,f,dedup_ECE_9_2025-12-31,dedup_ECE_9_2025-12-31_Germany,False,2025-12-31 19:57:00+00:00,420.0,True,True,True,False,False,False,False
78855324,2025-12-31,DB,ECE,international,Germany,No,Yes,No,DB,DB,DB,intermediate,f,dedup_ECE_9_2025-12-31,dedup_ECE_9_2025-12-31_Germany,False,2025-12-31 20:23:00+00:00,180.0,False,False,False,False,False,False,False
78855325,2025-12-31,DB,ECE,international,Germany,No,Yes,No,DB,DB,DB,intermediate,f,dedup_ECE_9_2025-12-31,dedup_ECE_9_2025-12-31_Germany,False,2025-12-31 20:58:00+00:00,0.0,False,False,False,False,False,False,False


In [5]:
db_measured = data_chuuchuu[
    (data_chuuchuu["operator_for_punctuality"] == "DB")
    & (data_chuuchuu["arrivalCancelled_resolved"] == "f")
    & data_chuuchuu["arrivalDelay"].notna()   # stops with an arrival delay measured
]

db_delayed_6 = ((db_measured['delayed_6min'].mean()) * 100)
db_punctuality_6 = ((1-db_measured['delayed_6min'].mean()) * 100)

print(f"{db_delayed_6:.2f}% of DB stops delayed > 6 min")
print(f"{db_punctuality_6:.2f}% of DB stops on time > 6 min")
print(f"{db_measured['delayed_6min'].sum()} delayed / {len(db_measured)} stops")

41.18% of DB stops delayed > 6 min
58.82% of DB stops on time > 6 min
1213419 delayed / 2946528 stops


# Eurostar

Eurostar is likely using the following parameters to calculate punctuality:

* Services: geographical groups of routes (Routes Belgium, London routes, Continental routes)
* Threshold (min): 6 (Belgium), 5 and 15 (London), 15 (Continental)
* Measurement: likely at terminus
* Train or passenger punctuality: train punctuality
* Account for cancelled trains: tested here as yes for London and Continental routes (a train with at least one cancelled stop, fully or partially cancelled, counts as delayed), no for Belgium
* Account joint services: to ask

How the route groups are built here:

* **Routes Belgium**: measured at stop level, on every Belgian station Eurostar calls at (arrivals only : not cancelled, delay measured)
* **London routes** (measured at terminus, one row per train, cancelled trains counted as delayed): London St Pancras is Eurostar's only UK station in the data, so a train touches London when it has a stop in the United Kingdom. Two versions, since we don't know which one Eurostar uses:
  * *opt 1, only arriving St Pancras*: the terminus is St Pancras
  * *opt 2, arriving + departing St Pancras*: St Pancras is the terminus or the departure (a train leaving London is measured at its continental terminus)
* **Continental routes** (measured at terminus, cancelled trains counted as delayed): trains with no stop in the United Kingdom
* **Continental routes, stop level** (measured on every arrival station, not only the terminus):
  * *all stops of non-London routes*: every station of the trains with no stop in the United Kingdom
  * *all stops except St Pancras*: every station of every train (UK routes included), St Pancras excluded
  * cancellation, two versions: *full + partial* = every cancelled stop counts as delayed; *full cancellation only* = only the stops of fully cancelled trains count as delayed, the cancelled stops of partially cancelled trains are left out

In [6]:
eurostar = data_chuuchuu[data_chuuchuu["operator_for_punctuality"] == "Eurostar"]

# journeys touching London (St Pancras = Eurostar's only UK station)
london_journeys = set(eurostar.loc[eurostar["country"] == "United Kingdom", "journey_id"])

# train cancellation: partial = at least one stop cancelled, full = every stop cancelled
share_stops_cancelled = (eurostar["arrivalCancelled_resolved"] == "t").groupby(eurostar["journey_id"]).mean()
cancelled_journeys = set(share_stops_cancelled.index[share_stops_cancelled > 0])
fully_cancelled_journeys = set(share_stops_cancelled.index[share_stops_cancelled == 1])

# STOP LEVEL: every arrival station (a train's departure station has no arrival, so it's left out),
# kept when it can be judged: cancelled, or arrival delay measured. is_cancelled = this stop is cancelled
eurostar_stops = eurostar[eurostar["depart_terminus"] != "depart"]
eurostar_stops = eurostar_stops.assign(
    is_cancelled=eurostar_stops["arrivalCancelled_resolved"] == "t",
    train_fully_cancelled=eurostar_stops["journey_id"].isin(fully_cancelled_journeys),
)
eurostar_stops = eurostar_stops[eurostar_stops["is_cancelled"] | eurostar_stops["arrivalDelay"].notna()]

eurostar_stops_served = eurostar_stops[~eurostar_stops["is_cancelled"]]  # cancellations not counted
# only full cancellation counted: the cancelled stops of partially cancelled trains are left out
eurostar_stops_full_only = eurostar_stops[~eurostar_stops["is_cancelled"] | eurostar_stops["train_fully_cancelled"]]

# TRAIN LEVEL: one row per train, its terminus. Kept when it can be judged: cancelled, or arrival
# delay measured at the terminus. is_cancelled = the train is fully or partially cancelled
eurostar_terminus = eurostar[eurostar["depart_terminus"] == "terminus"]
eurostar_terminus = eurostar_terminus.assign(
    is_cancelled=eurostar_terminus["journey_id"].isin(cancelled_journeys),
    train_fully_cancelled=eurostar_terminus["journey_id"].isin(fully_cancelled_journeys),
)
eurostar_terminus = eurostar_terminus[eurostar_terminus["is_cancelled"] | eurostar_terminus["arrivalDelay"].notna()]

is_london_stop = eurostar_stops["journey_id"].isin(london_journeys)
is_london_stop_full_only = eurostar_stops_full_only["journey_id"].isin(london_journeys)

# route group -> (where it is measured, how cancellations are counted, rows measured);
# a row counts as late when delayed by more than the threshold OR is_cancelled
eurostar_routes = {
    "Routes Belgium":
        ("all BE stops", "not counted", eurostar_stops_served[eurostar_stops_served["country"] == "Belgium"]),
    "London routes - opt 1 (only arriving St Pancras)":
        ("terminus", "train cancelled (full + partial)", eurostar_terminus[eurostar_terminus["country"] == "United Kingdom"]),
    "London routes - opt 2 (arriving + departing St Pancras)":
        ("terminus", "train cancelled (full + partial)", eurostar_terminus[eurostar_terminus["journey_id"].isin(london_journeys)]),
    "Continental routes":
        ("terminus", "train cancelled (full + partial)", eurostar_terminus[~eurostar_terminus["journey_id"].isin(london_journeys)]),
    "Continental routes - all stops of non-London routes":
        ("all stops", "stop cancelled (full + partial)", eurostar_stops[~is_london_stop]),
    "Continental routes - all stops except St Pancras":
        ("all stops except St Pancras", "stop cancelled (full + partial)", eurostar_stops[eurostar_stops["country"] != "United Kingdom"]),
    "Continental routes - all stops of non-London routes (full cancellation only)":
        ("all stops", "stop of a fully cancelled train", eurostar_stops_full_only[~is_london_stop_full_only]),
    "Continental routes - all stops except St Pancras (full cancellation only)":
        ("all stops except St Pancras", "stop of a fully cancelled train", eurostar_stops_full_only[eurostar_stops_full_only["country"] != "United Kingdom"]),
}

# (route group, threshold in min)
EUROSTAR_CASES = [
    ("Routes Belgium", 6),
    ("London routes - opt 1 (only arriving St Pancras)", 5),
    ("London routes - opt 2 (arriving + departing St Pancras)", 5),
    ("London routes - opt 1 (only arriving St Pancras)", 15),
    ("London routes - opt 2 (arriving + departing St Pancras)", 15),
    ("Continental routes", 15),
    ("Continental routes - all stops of non-London routes", 15),
    ("Continental routes - all stops except St Pancras", 15),
    ("Continental routes - all stops of non-London routes (full cancellation only)", 15),
    ("Continental routes - all stops except St Pancras (full cancellation only)", 15),
]

eurostar_results = []
for route, threshold in EUROSTAR_CASES:
    measured_at, cancellation_rule, rows = eurostar_routes[route]
    late = rows[f"delayed_{threshold}min"] | rows["is_cancelled"]
    eurostar_results.append({
        "services": route,
        "threshold_min": threshold,
        "measured_at": measured_at,
        "cancellation_counted_as_delayed": cancellation_rule,
        "n_measured": len(rows),  # stops for stop-level groups, trains for "terminus"
        # cancelled rows, split by whether their train is partially or fully cancelled
        # (trains for "terminus", stops for stop-level groups; 0 where cancellations are left out)
        "n_cancelled_partial": (rows["is_cancelled"] & ~rows["train_fully_cancelled"]).sum(),
        "n_cancelled_full": (rows["is_cancelled"] & rows["train_fully_cancelled"]).sum(),
        "n_cancelled_full_or_partial": rows["is_cancelled"].sum(),
        "n_delayed": late.sum(),  # delayed + cancelled
        "punctuality_chuuchuu_%": round((1 - late.mean()) * 100, 1),
    })
eurostar_results = pd.DataFrame(eurostar_results)

display(eurostar_results)

,services,threshold_min,measured_at,cancellation_counted_as_delayed,n_measured,n_cancelled_partial,n_cancelled_full,n_cancelled_full_or_partial,n_delayed,punctuality_chuuchuu_%
0,Routes Belgium,6,all BE stops,not counted,34938,0,0,0,7949,77.2
1,London routes - opt 1 (only arriving St Pancras),5,terminus,train cancelled (full + partial),9221,212,211,423,2356,74.4
2,London routes - opt 2 (arriving + departing St...,5,terminus,train cancelled (full + partial),18333,328,393,721,4085,77.7
3,London routes - opt 1 (only arriving St Pancras),15,terminus,train cancelled (full + partial),9221,212,211,423,1626,82.4
4,London routes - opt 2 (arriving + departing St...,15,terminus,train cancelled (full + partial),18333,328,393,721,2772,84.9
5,Continental routes,15,terminus,train cancelled (full + partial),24330,2303,1366,3669,7403,69.6
6,Continental routes - all stops of non-London r...,15,all stops,stop cancelled (full + partial),82232,4478,2733,7211,19739,76.0
7,Continental routes - all stops except St Pancras,15,all stops except St Pancras,stop cancelled (full + partial),101917,4856,3024,7880,22302,78.1
8,Continental routes - all stops of non-London r...,15,all stops,stop of a fully cancelled train,77754,0,2733,2733,15261,80.4
9,Continental routes - all stops except St Pancr...,15,all stops except St Pancras,stop of a fully cancelled train,97061,0,3024,3024,17446,82.0


<!--
## Punctuality units

Same logic as Step 7 of `Chuuchuu_pipeline.ipynb`, run on the full year: stops are collapsed into punctuality units (one per national segment for incumbents, one per whole journey for `operator_exception` operators), then each unit gets its cancellation status and its arrival outcome at the closing stop (terminus, or last stop before the border).
-->

# Flixtrain

In [7]:
# No official data for 2025 to compare to